In [1]:
import hashlib

#створюємо клас BloomFilter
class BloomFilter:
    def __init__(self, size=1000, num_hashes=3):
        #перевіряємо введені дані
        if not isinstance(size, int):
            raise TypeError('Розмір повинен бути числом')
        if not isinstance(num_hashes, int):
            raise TypeError('Кількість хешів повинна бути числом')
        if size <= 0:
            raise ValueError('Розмір повинен бути більше 0')
        if num_hashes <= 0:
            raise ValueError('Кількість хешів повинна бути більше 0')

        self.size = size
        self.num_hashes = num_hashes
        self.bits = bytearray(size)

    def _get_hashes(self, value):
        #отримуємо хеші
        hashes = []

        for i in range(self.num_hashes):
            data = (str(i) + value).encode('utf-8')
            hash_value = int(hashlib.md5(data).hexdigest(), 16)
            hashes.append(hash_value % self.size)

        return hashes

    def add(self, value):
        #перевіряємо дані
        if not isinstance(value, str):
            raise TypeError('Пароль повинен бути рядком')
        if len(value) == 0:
            raise ValueError('Пароль не може бути пустим')

        #додаємо пароль
        for index in self._get_hashes(value):
            self.bits[index] = 1

    def __contains__(self, value):
        #перевіряємо дані
        if not isinstance(value, str):
            return False
        if len(value) == 0:
            return False

        #перевіряємо пароль
        for index in self._get_hashes(value):
            if self.bits[index] == 0:
                return False

        return True


#функція для перевірки паролів
def check_password_uniqueness(bloom, passwords):
    #перевіряємо дані
    if not isinstance(bloom, BloomFilter):
        raise TypeError('Потрібно передати BloomFilter')
    if not isinstance(passwords, list):
        raise TypeError('Потрібно передати список паролів')

    results = {}

    #перевіряємо всі паролі
    for password in passwords:
        if not isinstance(password, str):
            results[password] = 'некоректне значення'
        elif len(password) == 0:
            results[password] = 'порожній пароль'
        elif password in bloom:
            results[password] = 'вже використаний'
        else:
            results[password] = 'унікальний'

    return results


if __name__ == '__main__':

    #створюємо фільтр Блума
    bloom = BloomFilter(size=1000, num_hashes=3)

    #додаємо існуючі паролі
    existing_passwords = [
        'password123',
        'admin123',
        'qwerty123']

    for password in existing_passwords:
        bloom.add(password)

    #перевіряємо нові паролі
    new_passwords_to_check = [
        'password123',
        'newpassword',
        'admin123',
        'guest']

    results = check_password_uniqueness(
        bloom,
        new_passwords_to_check)

    #виводимо результати
    for password, status in results.items():
        print(f"Пароль '{password}' - {status}.")
    print('--------------------------\n')

    #перевіряємо правильність результатів
    assert 'password123' in bloom
    assert 'admin123' in bloom
    assert 'newpassword' not in bloom
    assert results['password123'] == 'вже використаний'
    assert results['newpassword'] == 'унікальний'

    print('Перевірка пройдена')

Пароль 'password123' - вже використаний.
Пароль 'newpassword' - унікальний.
Пароль 'admin123' - вже використаний.
Пароль 'guest' - унікальний.
--------------------------

Перевірка пройдена


In [ ]:
import hashlib
import time


#створюємо клас HyperLogLog
class HyperLogLog:
    def __init__(self, num_registers=1024):
        #перевіряємо введені дані
        if not isinstance(num_registers, int):
            raise TypeError('Кількість регістрів повинна бути числом')
        if num_registers <= 0:
            raise ValueError('Кількість регістрів повинна бути більше 0')

        self.num_registers = num_registers
        self.registers = [0] * num_registers

    def add(self, value):
        #перевіряємо дані
        if not isinstance(value, str):
            raise TypeError('IP-адреса повинна бути рядком')
        if len(value) == 0:
            raise ValueError('IP-адреса не може бути пустою')

        #отримуємо хеш
        hash_value = int(
            hashlib.sha1(value.encode('utf-8')).hexdigest(),
            16
        )

        #визначаємо номер регістра
        register = hash_value % self.num_registers
        remaining = hash_value // self.num_registers

        #рахуємо кількість нулів
        if remaining == 0:
            rank = 64
        else:
            rank = 1
            while remaining & 1 == 0:
                rank += 1
                remaining = remaining >> 1

        #зберігаємо найбільше значення
        if rank > self.registers[register]:
            self.registers[register] = rank

    def count(self):
        #рахуємо кількість регістрів
        m = self.num_registers

        #рахуємо суму
        total = 0
        for register in self.registers:
            total += 2 ** (-register)

        #формула HyperLogLog
        result = 0.7213 / (1 + 1.079 / m)
        result = result * m * m / total

        return result


#функція для читання IP-адрес
def get_ip_addresses(filename):
    #перевіряємо дані
    if not isinstance(filename, str):
        raise TypeError('Назва файлу повинна бути рядком')
    if len(filename) == 0:
        raise ValueError('Назва файлу не може бути пустою')

    ips = []

    #відкриваємо файл
    with open(filename, 'r', encoding='utf-8') as file:

        #читаємо рядки
        for line in file:
            parts = line.split()

            if len(parts) > 0:
                ips.append(parts[0])

    return ips


#функція для точного підрахунку
def exact_count(ips):
    #перевіряємо дані
    if not isinstance(ips, list):
        raise TypeError('Потрібно передати список IP-адрес')

    unique_ips = set()

    #додаємо IP-адреси
    for ip in ips:
        if isinstance(ip, str) and len(ip) > 0:
            unique_ips.add(ip)

    return len(unique_ips)


#функція для наближеного підрахунку
def hll_count(ips):
    #перевіряємо дані
    if not isinstance(ips, list):
        raise TypeError('Потрібно передати список IP-адрес')

    hll = HyperLogLog(1024)

    #додаємо IP-адреси
    for ip in ips:
        if isinstance(ip, str) and len(ip) > 0:
            hll.add(ip)

    return hll.count()


if __name__ == '__main__':

    #назва лог-файлу
    filename = '1ms-stage-access.log'

    #завантажуємо IP-адреси
    ips = get_ip_addresses(filename)

    #точний підрахунок
    start_time = time.time()
    exact_result = exact_count(ips)
    exact_time = time.time() - start_time

    #наближений підрахунок
    start_time = time.time()
    hll_result = hll_count(ips)
    hll_time = time.time() - start_time

    #виводимо результати
    print('Результати порівняння:')
    print('                       Точний підрахунок   HyperLogLog')
    print(
        f'Унікальні елементи              '
        f'{exact_result}          {hll_result:.0f}'
    )
    print(
        f'Час виконання (сек.)             '
        f'{exact_time:.2f}          {hll_time:.2f}'
    )

    print('--------------------------\n')

    #перевіряємо правильність результатів
    assert exact_result >= 0
    assert hll_result >= 0
    assert exact_time >= 0
    assert hll_time >= 0

    print('Перевірка пройдена')

FileNotFoundError: [Errno 2] No such file or directory: '1ms-stage-access.log'